In [1]:
import pandas as pd

In [2]:
fund_path="../../data/dropbox/LaborDivision/Data/rawdata/article_fundagency.txt"
funder_cntry_path="../../data/dropbox/LaborDivision/Data/additionaldata/funder_country_new.csv"
pub_info_path="../matching/paper_fixed_data.csv"
cntry_path = '../../data/dropbox/LaborDivision/Data/additionaldata/cntry_meta.csv'

In [36]:
pub_funding_path="../../data/dropbox/LaborDivision/Data/deriveddata/funding/pub_full_funding.pkl"

In [3]:
pub_df=pd.read_csv(pub_info_path)
pub_df.head()

,Unnamed: 0,id,year,dis,cluster_id,role,Code,gender,yfp,npub,F,L,M,income_ni
0,0,29768070,2008,51,20009795,F,JPN,F,2008,5,2,0,0,H
1,1,31240446,2008,51,20009795,F,JPN,F,2008,5,2,0,0,H
2,2,29768070,2008,51,29411586,M,JPN,M,2000,74,3,0,5,H
3,3,31240446,2008,51,29411586,M,JPN,M,2000,74,3,0,5,H
4,4,43269102,2015,51,29411586,M,JPN,M,2000,74,3,0,5,H


In [4]:
pub_df=pub_df[['id','year','dis']].drop_duplicates()

In [5]:
pub_df=pub_df[pub_df.year>2008]
pub_df=pub_df[pub_df.dis<=100]

In [6]:
pub_df.head()

,id,year,dis
4,43269102,2015,51
5,45225060,2016,58
6,46664474,2016,18
18,33886678,2010,69
23,33150738,2009,26


In [7]:
fund_df=pd.read_csv(fund_path,sep=";",error_bad_lines=False, warn_bad_lines=False)
fund_df=fund_df.rename(columns={'OST_BK':'id','Agency':'agency'})
fund_df.head()

,id,agency
0,83622403,EPA Ireland
1,83622403,United States Environmental Protection Agency
2,83622403,EU
3,83622403,European Commission
4,83622403,European Space Agency


In [8]:
#subselect the funding information only for publications after 2008 and discipline smaller then 100
fund_df=fund_df[fund_df.id.isin(pub_df.id)]

In [9]:
funder_cntry=pd.read_csv(funder_cntry_path)[['label','agency','cntry']]
funder_cntry.head()

,label,agency,cntry
0,p1,National Natural Science Foundation of China,China
1,p1,National Institutes of Health,United States
2,p1,NIH,United States
3,p1,National Science Foundation,United States
4,p1,NSF,United States


In [10]:
fund_df=fund_df.merge(funder_cntry[['agency','cntry']],on='agency',how='left')

In [11]:
pub_miss_fund=fund_df[fund_df.cntry.isna()]['id'].unique()# article ids that have missing country information
#article rows that each of the funding institution has country
pub_with_fund=fund_df[~fund_df.id.isin(pub_miss_fund)] 

In [12]:
cntry_code=pd.read_csv(cntry_path)
p1=cntry_code[['cntry','Code']]
p2=cntry_code[['WoS','Code']].rename(columns={'WoS':'cntry'})
cntry_code=pd.concat([p1,p2])
cntry_code=cntry_code.drop_duplicates()
new_row={'cntry':'EU','Code':'EU'}
cntry_code=cntry_code.append(new_row, ignore_index=True)

In [13]:
cntry_code.head()

,cntry,Code
0,Afghanistan,AFG
1,Albania,ALB
2,Algeria,DZA
3,Andorra,AND
4,Angola,AGO


In [14]:
pub_with_fund=pub_with_fund.merge(cntry_code,on='cntry',how='left')

In [15]:
pub_with_fund=pub_with_fund.dropna()

In [16]:
pub_with_fund.head()

,id,agency,cntry,Code
0,83622403,EPA Ireland,Ireland,IRL
1,83622403,United States Environmental Protection Agency,United States,USA
2,83622403,EU,EU,EU
3,83622403,European Commission,EU,EU
4,83622403,European Space Agency,EU,EU


In [33]:
pub_with_fund=pub_with_fund[['id','Code']].drop_duplicates()
pub_with_fund=pub_with_fund.groupby('id')['Code'].apply(list)

In [37]:
pub_with_fund.reset_index().to_pickle(pub_funding_path)